# Silver — Orders (SCD1)
**GlobalMart Orchestration Lab**

| | |
|---|---|
| **Source** | `{catalog}.bronze.orders` |
| **Target** | `{catalog}.silver.orders` |
| **SCD Type** | SCD1 — orders are transactional, update in place |
| **Depends on** | Bronze Orders **and** Silver Customers (referential-integrity check) |

## Step 1 — Setup

In [ ]:
from pyspark.sql.functions import col, row_number, desc, initcap, to_date, current_timestamp, when, lit
from pyspark.sql.window import Window
from delta.tables import DeltaTable

dbutils.widgets.text('catalog',       'your_catalog')
dbutils.widgets.text('source_schema', 'bronze')
dbutils.widgets.text('target_schema', 'silver')

CATALOG          = dbutils.widgets.get('catalog')
SOURCE_SCHEMA    = dbutils.widgets.get('source_schema')
TARGET_SCHEMA    = dbutils.widgets.get('target_schema')

SOURCE_TABLE     = f'{CATALOG}.{SOURCE_SCHEMA}.orders'
SILVER_CUSTOMERS = f'{CATALOG}.{TARGET_SCHEMA}.customers'
TABLE            = f'{CATALOG}.{TARGET_SCHEMA}.orders'

print(f'Source: {SOURCE_TABLE}')
print(f'Target: {TABLE}')

## Step 2 — Read &amp; Deduplicate from Bronze

In [ ]:
bronze_df = spark.table(SOURCE_TABLE)
print(f"Bronze rows (all batches): {bronze_df.count():,}")

dedup_window = Window.partitionBy("order_id").orderBy(desc("_ingested_at"))
deduped_df = (
    bronze_df
    .withColumn("_rn", row_number().over(dedup_window))
    .filter(col("_rn") == 1)
    .drop("_rn")
)
print(f"After dedup (latest per order_id): {deduped_df.count():,}")

## Step 3 — DQ Scan
**Status standardization:** source data mixes casing (`delivered`, `PLACED`, `Shipped`) — `initcap()` normalizes it.
**Referential integrity:** an order whose `customer_id` doesn't exist in `silver.customers` can't be trusted downstream — Gold's join would silently produce a row with no real customer behind it.

In [ ]:
cleaned_df = (
    deduped_df
    .withColumn("status",     initcap(col("status")))
    .withColumn("order_date", to_date(col("order_date")))
    .withColumn("created_at", to_date(col("created_at")))
)

known_customers = spark.table(SILVER_CUSTOMERS).select("customer_id").distinct()

dq_df = cleaned_df.join(
    known_customers.withColumnRenamed("customer_id", "_known_customer_id"),
    cleaned_df.customer_id == col("_known_customer_id"),
    "left"
).withColumn(
    "_dq_issue",
    when(col("order_id").isNull(),                lit("NULL_ORDER_ID"))
    .when(col("_known_customer_id").isNull(),      lit("ORPHANED_CUSTOMER_ID"))
    .otherwise(lit(None))
).drop("_known_customer_id")

dq_df.groupBy("_dq_issue").count().orderBy(desc("count")).display()

## Step 4 — Decision Per Issue

| Issue | Decision | Why |
|---|---|---|
| `NULL_ORDER_ID` | **Quarantine** | No usable key |
| `ORPHANED_CUSTOMER_ID` | **Quarantine** | Can't attribute the order to a real customer — unusable for Gold's join, and the missing customer might just not have loaded yet (arrives in a later batch) |

In [ ]:
quarantine_df = dq_df.filter(col("_dq_issue").isNotNull())
clean_df      = dq_df.filter(col("_dq_issue").isNull()).drop("_dq_issue")

print(f"Quarantined : {quarantine_df.count():,}")
print(f"Valid orders: {clean_df.count():,}")
if quarantine_df.count() > 0:
    quarantine_df.select("order_id", "customer_id", "_dq_issue").display()

## Step 5 — Create Silver Table (first run only)

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{TARGET_SCHEMA}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLE} (
        order_id           STRING,
        customer_id        STRING,
        order_date         DATE,
        status             STRING,
        created_at         DATE,
        _silver_updated_at TIMESTAMP
    )
    USING DELTA
""")
print(f"Table ready: {TABLE}")

## Step 6 — SCD1 MERGE
Update the row if `order_id` already exists (status may have changed), insert if new. Safe to re-run.

In [ ]:
silver_df = clean_df.withColumn("_silver_updated_at", current_timestamp()) \
    .select("order_id", "customer_id", "order_date", "status", "created_at", "_silver_updated_at")

target = DeltaTable.forName(spark, TABLE)

(target.alias("t")
    .merge(silver_df.alias("s"), "t.order_id = s.order_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
print("MERGE complete")

## Step 7 — Verify

In [ ]:
result = spark.table(TABLE)
print(f"Total rows in {TABLE}: {result.count():,}")
print("Status distribution:")
result.groupBy("status").count().display()
result.display()